# Generate 1024 parameter sets to effectively sample the parameter space of WOMBAT-mid run in RYF of ACCESS-OM2

### imports

In [43]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sci
from scipy.stats import qmc

wrkdir = "/g/data/vn19/pjb581/SOTS-Optimization/WOMBATfull/data"

# print versions of packages
print("python version =",sys.version[:5])
print("numpy version =", np.__version__)
print("pandas version =", pd.__version__)
print("scipy version =", sci.__version__)
print("matplotlib version =", sys.modules[plt.__package__].__version__)

os.chdir(wrkdir)
os.listdir()


python version = 3.10.
numpy version = 1.26.4
pandas version = 2.3.3
scipy version = 1.15.3
matplotlib version = 3.10.6


['BIAS_optimal.xlsx',
 'BIAS.xlsx',
 'GPR_pco2_cost.joblib',
 'GPR_pco2_rmse.joblib',
 'NSDEV.xlsx',
 'NRMSE_optimal.xlsx',
 'parameter_ranges.xlsx',
 'GPR_chl_bottle_corr.joblib',
 'GPR_bac_corr.joblib',
 'parameter_norm_optimal100_2.txt',
 'parameter_sets_optimal100_2.txt',
 'GPR_micro_corr.joblib',
 'SDEV_opt.xlsx',
 'optimisation_norm_2048.txt',
 'GPR_micro_cost.joblib',
 'GPR_chl_bottle_cost.joblib',
 'parameter_NRMSEpredicted_optimal100_2.txt',
 'apriori_parameter_ranges.xlsx',
 'GPR_fgco2_rmse.joblib',
 'parameter_norm_optimal100_3.txt',
 'CCOEF_opt.xlsx',
 'GPR_no3.joblib',
 'GPR_poc_cost.joblib',
 'GPR_sil.joblib',
 'GPR_sil_cost.joblib',
 'GPR_fgco2.joblib',
 'SDEV.xlsx',
 'GPR_pco2.joblib',
 'mcmc_wombat_mid_26params_1M_rmse_withSIL.h5',
 'CCOEF_optimal.xlsx',
 'mcmc_wombat_mid_26params_1M_rmse.h5',
 'GPR_sil_rmse.joblib',
 'GPR_fgco2_cost.joblib',
 'NSDEV_optimal.xlsx',
 'GPR_chl_bottle_rmse.joblib',
 'GPR_b1mb2.joblib',
 'NRMSE.xlsx',
 'CCOEF.xlsx',
 'GPR_nh4_corr.joblib',

### find the Sobol parameter sequence for the complete parameter set


In [44]:
df = pd.read_excel("apriori_parameter_ranges_SOTSexps.xlsx", sheet_name="parameter_ranges")
df

,WOMBAT-full,min,max,units,References,sensitivity analysis?,optimisation?,parameter number,optim number
0,alphabio_phy,1,4,(w/m2)-1 (mg Chl / mg C)-1,"MacIntyre et al., 2002",1,1,1,1
1,abioa_phy,0.0000028935185185,0.0000173611111111,/s,"Anderson et al., 2021 Nature Communications",1,1,2,2
2,bbioa_phy,1.04,1.0800000000000001,none,"Anderson et al., 2021 Nature Communications",1,1,3,3
3,alphabio_dia,1,4,(w/m2)-1 (mg Chl / mg C)-1,"MacIntyre et al., 2002; Edwards et al., 2015; ...",1,1,4,4
4,alphabio_tri,0.25,2,(w/m2)-1 (mg Chl / mg C)-1,"Masotti et al., 2007 Marine Ecology Progress S...",1,0,5,4
...,...,...,...,...,...,...,...,...,...
149,bac_C2Fe,1/40e-6,1/40e-6,mol C / mol Fe,Fourquez et al. 2020,0,0,85,33
150,baclmor,-4,-1,/s,Baker & Geider 2021,1,1,86,34
151,bacqmor,0.0000001157407407,0.0000028935185185,(mmol C / m3)-1 day-1,Suttle 1994,1,0,87,34
152,aox_knh4,0.45,0.45,mmol NH4 / m3,"Awata et al., 2013",0,0,87,34


### select only the parameters we are going to vary and collect information

In [45]:
param_ranges = df[df["sensitivity analysis?"]==1]

param_ranges = param_ranges.drop(["units", "References", "sensitivity analysis?", "optimisation?"], axis=1)
param_ranges = param_ranges.set_index("WOMBAT-full")
param_ranges



,min,max,parameter number,optim number
WOMBAT-full,,,,
alphabio_phy,1,4,1,1
abioa_phy,0.0000028935185185,0.0000173611111111,2,2
bbioa_phy,1.04,1.0800000000000001,3,3
alphabio_dia,1,4,4,4
alphabio_tri,0.25,2,5,4
...,...,...,...,...
obac_kdoc,1,200,83,32
obac_fele,0.01,0.4,84,32
sbac_kdoc,1,200,85,33


In [46]:
pd.set_option("display.precision", 16)
param_ranges.loc['abioa_phy']

min                 0.0000028935185185
max                 0.0000173611111111
parameter number                     2
optim number                         2
Name: abioa_phy, dtype: object

### Create a normalized sobol sample

In [47]:
dim = len(param_ranges)
exps = 2048
sampler = qmc.Sobol(d=dim, scramble=True, seed=10)
sample_qmc = sampler.random(n=exps)  #here, n=256=2^8 
sample_qmc.shape


(2048, 87)

### Create the parameter sets for our sensitivity experiments

In [48]:
param_sets = np.zeros((exps,dim))

# Loop over variables correctly
for ii, var in enumerate(param_ranges.index):
    param_sets[:, ii] = (
        param_ranges.loc[var, "min"]
        + (param_ranges.loc[var, "max"] - param_ranges.loc[var, "min"])
        * sample_qmc[:, ii]
    )

names = param_ranges.index
df_exps = pd.DataFrame(param_sets, columns=names)
df_norm = pd.DataFrame(sample_qmc, columns=names)
df_norm


WOMBAT-full,alphabio_phy,abioa_phy,bbioa_phy,alphabio_dia,alphabio_tri,abioa_dia,bbioa_dia,bbioh,phykn,phykf,...,aoalmor,pbac_alpha,lbac_kdoc,lbac_alpha,lbac_beta,obac_kdoc,obac_fele,sbac_kdoc,baclmor,bacqmor
0,0.9870995162054896,0.8023925330489874,0.0114757269620895,0.1757747754454613,0.8299645446240902,0.0203756187111139,0.0286230696365237,0.3057573009282351,0.7182503324002028,0.0698164757341146,...,0.6666613603010774,0.2629278162494302,0.9615393644198775,0.7588808247819543,0.0115507394075394,0.9737481260672212,0.2422127621248364,0.1018263446167111,0.4989032158628106,0.3516822485253215
1,0.1040492448955774,0.0644684070721269,0.7750228000804782,0.5165035761892796,0.3321604970842600,0.6592275109142065,0.9213297599926591,0.5176461748778820,0.2263398328796029,0.8218004172667861,...,0.2604350931942463,0.6586275855079293,0.4466942958533764,0.4170716991648078,0.6603968665003777,0.2145928181707859,0.8121762061491609,0.6813819548115134,0.8519640415906906,0.5240224841982126
2,0.4719106759876013,0.6062689293175936,0.3685875535011292,0.4949254887178540,0.1701297173276544,0.8700682483613491,0.3743643388152122,0.9377531977370381,0.3901377283036709,0.5829852065071464,...,0.8722903588786721,0.1784500870853662,0.2386093009263277,0.1872379323467612,0.9707872020080686,0.6828401312232018,0.4957399200648069,0.9982464481145144,0.6087066875770688,0.8563524503260851
3,0.6205953499302268,0.2768082888796926,0.6046785814687610,0.8377290153875947,0.6679292386397719,0.4503287374973297,0.7005884889513254,0.2297819880768657,0.9289279161021113,0.3378978800028563,...,0.2160628847777843,0.7752996776252985,0.7242508465424180,0.5290206233039498,0.3261525155976415,0.3804719941690564,0.5586500037461519,0.2966815754771233,0.2439954467117786,0.0277160061523318
4,0.6817727470770478,0.6643630778416991,0.9831169787794352,0.9095224235206842,0.6031528366729617,0.2122180769219995,0.4619437996298075,0.8060667226091027,0.3380427109077573,0.9656316544860601,...,0.1193818766623735,0.0803605709224939,0.1097795432433486,0.9182230401784182,0.4672183077782393,0.0970523348078132,0.2569142514839768,0.5688302079215646,0.2983075315132737,0.6493946332484484
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2043,0.6824112832546234,0.2742059724405408,0.3213151842355728,0.6302032070234418,0.1659136647358537,0.6459910133853555,0.4206063430756330,0.7628221260383725,0.7296256422996521,0.5348300170153379,...,0.3506911164149642,0.6396245099604130,0.0090567618608475,0.4463805230334401,0.9117518542334437,0.2708450071513653,0.3217084379866719,0.2873144401237369,0.0675718598067760,0.3256681459024549
2044,0.6206893380731344,0.6679233601316810,0.1997077772393823,0.6248761322349310,0.1051636384800076,0.8760524243116379,0.6824482660740614,0.2177859609946609,0.0060949828475714,0.1615184452384710,...,0.4376038694754243,0.4427322186529636,0.6528859566897154,0.1180525701493025,0.8029434978961945,0.2379292305558920,0.6117852833122015,0.5776516068726778,0.3883055457845330,0.9471456818282604
2045,0.4724890282377601,0.4651736468076706,0.9676510486751795,0.2156051788479090,0.6078638462349772,0.3034751825034618,0.2587203346192837,0.9883229667320848,0.5610556835308671,0.9175982987508178,...,0.5938606746494770,0.6035468317568302,0.1381602035835385,0.7137989401817322,0.4002464488148689,0.9502095384523273,0.4275907073169947,0.1547660455107689,0.7764540761709213,0.1810558335855603
2046,0.1038943259045482,0.9868306173011661,0.4357280293479562,0.8036363972350955,0.8949136957526207,0.2336332853883505,0.9616917343810201,0.5374504104256630,0.8347187526524067,0.6786002805456519,...,0.0229943254962564,0.1238897731527686,0.4221424413844943,0.9514218419790268,0.2187010841444135,0.4194639241322875,0.8611544193699956,0.4628395484760404,0.5275241481140256,0.4502878608182073


### Save the parameter sets to an excel spreadsheet

In [49]:
%%time
    
os.chdir(wrkdir)
df_exps.to_excel(wrkdir + '/parameter_sets_2048.xlsx')
df_exps.to_csv(wrkdir + '/parameter_sets_2048.txt', sep=' ')
df_norm.to_excel(wrkdir + '/parameter_norm_2048.xlsx')
df_norm.to_csv(wrkdir + '/parameter_norm_2048.txt', sep=' ')


CPU times: user 7.44 s, sys: 82.9 ms, total: 7.53 s
Wall time: 7.54 s


In [50]:
os.getcwd()

'/g/data/vn19/pjb581/SOTS-Optimization/WOMBATfull/data'